# U-Net 水下偏色图像增强 - Colab 训练配置和流程

这个 Notebook 用于将由于算力限制或本地开发完成的 U-Net 项目迁移到 Google Colab 的 GPU 上运行。

In [ ]:
# 1. 查看分配到的 GPU 资源
# 建议在菜单栏选择 "代码执行程序(Runtime)" -> "更改运行时类型" -> 硬件加速器选择 T4 GPU 或更高级 GPU
!nvidia-smi

In [ ]:
# 2. 挂载 Google Drive
# 我们将使用 Google Drive 来存放代码、数据集以及长期保存训练好的模型权重。
from google.colab import drive
drive.mount('/content/drive')

### 3. 项目和数据的准备 (通过 GitHub)

你可以直接从 GitHub 克隆你的代码仓库。为了避免每次重启 Colab 都需要重新克隆代码导致训练 Checkpoint 丢失，我们建议直接将仓库克隆到挂载好的 Google Drive 中。

In [ ]:
# 切换到 Google Drive 的云盘外层目录
%cd /content/drive/MyDrive

# 从你的 GitHub 仓库拉取代码 (请将下方 URL 替换为你自己真实的 GitHub 仓库地址！)
!git clone https://github.com/你的用户名/你的仓库名.git u-net

# 切换进项目工作目录
%cd /content/drive/MyDrive/u-net

# [可选] 如果之后你在本地更新了代码并 push 上去，你可以取消注释并运行下方命令拉取最新代码：
# !git pull

# 查看目录结构，确保 dataset.py, train.py, model.py 等都在这
!ls -al

In [ ]:
# 4. 安装本地缺少的依赖项
# Colab 默认已经自带大部分常用库 (PyTorch, torchvision, PIL)
!pip install -r requirements.txt

In [ ]:
# 5. 执行模型训练代码
# 请确保当前目录下面有 data 文件夹，并且里面放好了 train/ 和 val/ 的输入和目标图。
# num_workers 在 Colab 上建议使用 2。如果出现内存不足等问题，可以把 num_workers 改为 0。
!python train.py \
    --mode train \
    --train_input_dir data/train/input \
    --train_target_dir data/train/target \
    --val_input_dir data/val/input \
    --val_target_dir data/val/target \
    --checkpoint_dir checkpoints \
    --max_epochs 150 \
    --batch_size 8 \
    --num_workers 2

In [ ]:
# 6. 推理与验证 (训练完成后)
# 指定一张测试图像路径
!python train.py --mode infer \
    --input_img data/val/input/test.jpg \
    --output_img output_enhanced.png \
    --ckpt checkpoints/best_model.pth

In [ ]:
# 7. 可视化查看推理前后的对比结果
import matplotlib.pyplot as plt
from PIL import Image

try:
    img_in = Image.open('data/val/input/test.jpg') # 替换为你真实测试的图片名字
    img_out = Image.open('output_enhanced.png')
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_in)
    axes[0].set_title("Input Image (Degraded)")
    axes[0].axis("off")

    axes[1].imshow(img_out)
    axes[1].set_title("Enhanced Output")
    axes[1].axis("off")

    plt.show()
except FileNotFoundError:
    print("请确保上面指定的测试图片路径存在哦")

---
## 8. 模型指标评估 —— PSNR & SSIM

训练完毕后，运行下面的代码块对整个 **验证集** 进行批量评估，输出平均 PSNR 和 SSIM 分数。

- **PSNR（峰值信噪比）**：越高越好，单位 dB，20 dB 以上为良好
- **SSIM（结构相似度）**：越接近 1 越好，0.75 以上为良好

In [ ]:
# ── 8.1 导入所需库 ────────────────────────────────────────────────────────
import math
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from model import UNet

# ── 8.2 配置参数（根据你的实际情况修改这里）────────────────────────────────
VAL_INPUT_DIR  = 'data/val/input'       # 验证集退化图像目录
VAL_TARGET_DIR = 'data/val/target'      # 验证集参考标准图目录
CKPT_PATH      = 'checkpoints/best_model.pth'  # 模型权重路径
IMG_SIZE       = 256                    # 与训练时保持一致

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'运行设备: {device}')

In [ ]:
# ── 8.3 定义 PSNR 计算函数 ────────────────────────────────────────────────
def compute_psnr(pred, target, max_val=1.0):
    """
    计算 PSNR（峰值信噪比）。
    pred / target: [C, H, W]，值域 [0, 1]
    """
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float('inf')
    return 10 * math.log10((max_val ** 2) / mse)

print('✅ PSNR 函数已定义')

In [ ]:
# ── 8.4 定义 SSIM 计算函数（纯 PyTorch，无需额外安装库）─────────────────────
def _gaussian_kernel(size=11, sigma=1.5):
    coords = torch.arange(size, dtype=torch.float32) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    return g / g.sum()

def compute_ssim(pred, target, window_size=11, sigma=1.5, C1=0.01**2, C2=0.03**2):
    """
    计算 SSIM（结构相似度索引）。
    pred / target: [C, H, W]，值域 [0, 1]
    参考论文: Wang et al., IEEE TIP 2004
    """
    pred   = pred.unsqueeze(0)    # [1, C, H, W]
    target = target.unsqueeze(0)  # [1, C, H, W]
    C_ch   = pred.shape[1]

    kernel_1d = _gaussian_kernel(window_size, sigma).to(pred.device)
    kernel_2d = torch.outer(kernel_1d, kernel_1d)
    kernel    = kernel_2d.unsqueeze(0).unsqueeze(0).repeat(C_ch, 1, 1, 1)
    pad       = window_size // 2

    mu1 = F.conv2d(pred,   kernel, padding=pad, groups=C_ch)
    mu2 = F.conv2d(target, kernel, padding=pad, groups=C_ch)

    mu1_sq  = mu1 ** 2
    mu2_sq  = mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(pred * pred,     kernel, padding=pad, groups=C_ch) - mu1_sq
    sigma2_sq = F.conv2d(target * target, kernel, padding=pad, groups=C_ch) - mu2_sq
    sigma12   = F.conv2d(pred * target,   kernel, padding=pad, groups=C_ch) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean().item()

print('✅ SSIM 函数已定义')

In [ ]:
# ── 8.5 加载模型权重 ──────────────────────────────────────────────────────
model = UNet(in_channels=3, out_channels=3).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ 模型加载成功 ← Epoch {ckpt["epoch"]}，'
      f'最优验证损失: {ckpt.get("best_val_loss", "N/A")}')

In [ ]:
# ── 8.6 批量计算验证集的 PSNR & SSIM ──────────────────────────────────────
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # → [-1, 1]
])

input_dir  = Path(VAL_INPUT_DIR)
target_dir = Path(VAL_TARGET_DIR)
exts       = {'.jpg', '.jpeg', '.png', '.bmp'}
img_files  = sorted([f for f in input_dir.iterdir() if f.suffix.lower() in exts])

psnr_list, ssim_list = [], []

for inp_path in tqdm(img_files, desc='Evaluating'):
    tgt_path = target_dir / inp_path.name
    if not tgt_path.exists():
        continue

    inp_tensor = transform(Image.open(inp_path).convert('RGB')).unsqueeze(0).to(device)
    tgt_tensor = transform(Image.open(tgt_path).convert('RGB')).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_tensor = model(inp_tensor)

    # 反归一化到 [0, 1] 再计算
    pred_01   = ((pred_tensor.squeeze(0) + 1.0) / 2.0).clamp(0.0, 1.0)
    target_01 = ((tgt_tensor.squeeze(0)  + 1.0) / 2.0).clamp(0.0, 1.0)

    psnr_list.append(compute_psnr(pred_01, target_01))
    ssim_list.append(compute_ssim(pred_01, target_01))

avg_psnr = sum(psnr_list) / len(psnr_list)
avg_ssim = sum(ssim_list) / len(ssim_list)

print()
print('=' * 40)
print(f'  评估完成  共 {len(psnr_list)} 张图像')
print(f'  平均 PSNR : {avg_psnr:.4f} dB')
print(f'  平均 SSIM : {avg_ssim:.4f}')
print('=' * 40)

In [ ]:
# ── 8.7 用柱状图可视化全部验证图像的 PSNR 和 SSIM 分布 ─────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# PSNR 分布直方图
axes[0].hist(psnr_list, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(avg_psnr, color='red', linestyle='--', label=f'Mean: {avg_psnr:.2f} dB')
axes[0].set_title('PSNR Distribution (dB)')
axes[0].set_xlabel('PSNR (dB)')
axes[0].set_ylabel('Count')
axes[0].legend()

# SSIM 分布直方图
axes[1].hist(ssim_list, bins=20, color='darkorange', edgecolor='white')
axes[1].axvline(avg_ssim, color='red', linestyle='--', label=f'Mean: {avg_ssim:.4f}')
axes[1].set_title('SSIM Distribution')
axes[1].set_xlabel('SSIM')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('eval_metrics.png', dpi=150)
plt.show()
print('图表已保存为 eval_metrics.png')